# PlannedCross — brapi-py

Interactive notebook for exploring the BrAPI **PlannedCross** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Install / upgrade brapi-py — uncomment ONE of the options below:

# Option 1: install from PyPI (once the package is published)
# %pip install -q brapi-py

# Option 2: install from local source (for development / testing)
# import subprocess, sys, pathlib
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e',
#                        str(pathlib.Path('..').resolve())])

import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient
from dotenv import load_dotenv
import os

# Loads credentials from ../.env (never committed to git)
# Create a .env file in the project root with these keys:
#   BRAPI_BASE_URL=https://brapi.example.com
#   BRAPI_TOKEN_ENDPOINT=https://auth.example.com/token
#   BRAPI_CLIENT_ID=my-client
#   BRAPI_CLIENT_SECRET=my-secret
load_dotenv(dotenv_path='../.env')

client = BrapiClient(
    base_url=os.environ['BRAPI_BASE_URL'],
    token_endpoint=os.environ['BRAPI_TOKEN_ENDPOINT'],
    client_id=os.environ['BRAPI_CLIENT_ID'],
    client_secret=os.environ['BRAPI_CLIENT_SECRET'],
)
print('Client ready →', client)

## 3 — List  (`GET /plannedcrosses`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /plannedcrosses — list records
df = (
    client.planned_cross
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 4 — Single-record CRUD

Get, create, update, and delete individual records.

In [ ]:
from brapi.entities.generated_planned_cross import PlannedCross

# POST /plannedcrosses — create a new record
new_planned_cross = PlannedCross(
    plannedCrossDbId='',  # assigned by server
)
created = client.planned_cross.create(new_planned_cross)
print('Created plannedCrossDbId:', created.plannedCrossDbId)

In [ ]:
# PUT /plannedcrosses/ {id} — update the record created above
# (run the create cell first to populate 'created')
created.plannedCrossName = 'updated value'
updated = client.planned_cross.update(created.plannedCrossDbId, created)
print('Updated:', updated.plannedCrossDbId)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.planned_cross
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'plannedCrossName'
if 'plannedCrossName' in df.columns:
    df['plannedCrossName'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'externalReferences' JSON column into separate rows
col = 'externalReferences'
if col in df.columns:
    id_col = 'plannedCrossDbId'
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 7 — Error handling

In [ ]:
import httpx


# ValueError from create_many with only one item
try:
    client.planned_cross.create_many([])
except (ValueError, AttributeError) as e:
    print('Error:', e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')